In [5]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PARQUET = "../data/student_dataset/flows.parquet"   # path from scratch/; adapt if the notebook is elsewhere

con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '6GB'")

con.execute(f"""
CREATE OR REPLACE VIEW f AS
SELECT flow_id,
       CAST(flow_start AS TIMESTAMPTZ) AS t_start,
       CAST(flow_end   AS TIMESTAMPTZ) AS t_end,
       start_rtt, end_rtt, delta_rtt,
       "meta.src_ip"   AS src_ip,   "meta.dst_ip"   AS dst_ip,
       "meta.src_port" AS src_port, "meta.dst_port" AS dst_port,
       "meta.cric.src.name"   AS src_site, "meta.cric.dst.name" AS dst_site,
       "meta.application"     AS application,
       "meta.syslog.hostname" AS hostname
FROM read_parquet('{PARQUET}')
""")

SERVICE_PORTS = [1095, 1094, 2880, 443, 8443, 8446, 2881, 18443, 11000, 9000, 11001, 9001, 2882, 1101, 8001,
                 1102, 1103, 25684, 28714, 27647, 25328, 29326, 25464, 25365, 28922, 2889, 28908, 27323,
                 29626, 26974, 25708, 27146, 25315, 27661, 28835, 26770, 8459, 29128, 29125, 27618, 25392, 25473]
ports_sql = ", ".join(map(str, SERVICE_PORTS))

con.execute(f"""
CREATE OR REPLACE MACRO is_service(p) AS
    (p IN ({ports_sql}) OR p BETWEEN 20000 AND 24999 OR p BETWEEN 30000 AND 30999)
""")

con.execute("""
CREATE OR REPLACE VIEW fo AS
SELECT *,
       is_service(src_port) AS src_svc,
       is_service(dst_port) AS dst_svc,
       CASE WHEN is_service(dst_port) AND NOT is_service(src_port) THEN 'C2S'
            WHEN is_service(src_port) AND NOT is_service(dst_port) THEN 'S2C'
            WHEN is_service(src_port) AND is_service(dst_port)     THEN 'both_service'
            ELSE 'no_service' END AS orient_class
FROM f
""")

def q(sql, params=None):
    """Run a query and return the result as a DataFrame."""
    return con.execute(sql, params or []).df()

EX_COLS = "flow_id, t_start, src_ip, src_port, dst_ip, dst_port, application, hostname, src_site, dst_site"

def examples(where, n=3, extra="", params=None):
    """n reproducible example flows satisfying a condition (same order on every run)."""
    return q(f"""
        SELECT {EX_COLS}{extra}
        FROM fo
        WHERE {where}
        ORDER BY hash(flow_id || CAST(t_start AS VARCHAR))
        LIMIT {n}
    """, params)

## Q1. How are source and destination assigned?
**Question for the developers:** how are `src` and `dst` assigned in a Firefly event? Is the source always the host
that reports the flow, the host that opened the connection, or does it depend on the service (XRootD vs HTTP/WebDAV)?
Is there a field that states the role (client/server) of each endpoint?

**Evidence:** for each service port, the share of flows in which the service port is the *source* port
(i.e. the server is recorded as source).

In [6]:
q("""
SELECT CASE WHEN orient_class = 'C2S' THEN dst_port ELSE src_port END AS service_port,
       COUNT(*) AS n,
       ROUND(100 * AVG(CASE WHEN orient_class = 'S2C' THEN 1 ELSE 0 END), 1) AS pct_server_is_source
FROM fo
WHERE orient_class IN ('C2S', 'S2C')
GROUP BY 1 ORDER BY n DESC LIMIT 12
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,service_port,n,pct_server_is_source
0,1095,42040796,0.9
1,1094,17097017,25.2
2,2880,1626992,99.7
3,443,1363080,100.0
4,8443,709837,99.8
5,8446,294734,100.0
6,2881,165583,100.0
7,18443,153012,100.0
8,11000,128535,99.9
9,11001,73595,99.9


**Examples:** servers that appear in both conventions (as destination in some flows, as source in others),
with at least 1,000 flows of each kind.

In [7]:
both_ways = q("""
WITH s AS (
    SELECT CASE WHEN orient_class = 'C2S' THEN dst_ip ELSE src_ip END AS server_ip, orient_class
    FROM fo WHERE orient_class IN ('C2S', 'S2C'))
SELECT server_ip,
       COUNT(*) FILTER (WHERE orient_class = 'C2S') AS n_server_is_dst,
       COUNT(*) FILTER (WHERE orient_class = 'S2C') AS n_server_is_src
FROM s GROUP BY 1
HAVING n_server_is_dst >= 1000 AND n_server_is_src >= 1000
ORDER BY LEAST(n_server_is_dst, n_server_is_src) DESC
LIMIT 5
""")
both_ways

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,server_ip,n_server_is_dst,n_server_is_src
0,2001:630:54:21:82f6:b172::,107940,89352
1,2001:630:54:20:82f6:b152::,109176,89017
2,2001:630:54:22:82f6:b192::,108027,88796
3,2001:630:54:23:82f6:b1d2::,108551,88453
4,2001:630:54:1c:82f6:b132::,104206,85355


Two flows in which this server is the destination and two in which it is the source.

In [8]:
ip = both_ways["server_ip"].iloc[0]
pd.concat([
    examples("dst_ip = ? AND orient_class = 'C2S'", n=2, params=[ip]),
    examples("src_ip = ? AND orient_class = 'S2C'", n=2, params=[ip]),
], ignore_index=True)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()
/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


,flow_id,t_start,src_ip,src_port,dst_ip,dst_port,application,hostname,src_site,dst_site
0,3f66070d4bf702866fa4ede3cf5b49e289c376ef51012c...,2026-10-10 02:40:06.512626+00:00,2001:630:113::dc,57928,2001:630:54:21:82f6:b172::,1094,unknown,ceph-svc37.gridpp.rl.ac.uk,NaN,RAL-LCG2
1,5d63d7a117b523968bfc19e46514be8e6492c32e52164a...,2026-10-10 13:51:19.620318+00:00,2001:630:54:12:82f6:db6b::,46494,2001:630:54:21:82f6:b172::,1094,unknown,ceph-svc37.gridpp.rl.ac.uk,RAL-LCG2,RAL-LCG2
2,827aae00be5e49397cd15db5499347186786644fc26013...,2026-10-09 03:12:27.044554+00:00,2001:630:54:21:82f6:b172::,1094,2001:12d0:8120::104,47108,unknown,ceph-svc37.gridpp.rl.ac.uk,RAL-LCG2,SAMPA
3,894aa687087a0c2d5ea692ed2faa1a40d29d0acc421fc1...,2026-10-13 02:55:46.175141+00:00,2001:630:54:21:82f6:b172::,1094,2001:1458:301:10::100:cd,36822,http-put,st-096-hh15140e.cern.ch,RAL-LCG2,CERN-PROD


## Q2. Does the convention depend on the application or on the reporting host?
**Question for the developers:** does the src/dst convention depend on the application label or on the reporting
host (`syslog.hostname`)? Can the same application be reported with opposite conventions by different hosts?

**Evidence 1:** share of flows with the server recorded as source, per application.

In [9]:
q("""
SELECT application, COUNT(*) AS n,
       ROUND(100 * AVG(CASE WHEN orient_class = 'S2C' THEN 1 ELSE 0 END), 1) AS pct_server_is_source
FROM fo
WHERE orient_class IN ('C2S', 'S2C')
GROUP BY 1 ORDER BY n DESC LIMIT 15
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,application,n,pct_server_is_source
0,cmsRun,31289348,0.0
1,http-put,8836756,100.0
2,JobWrapper,8727865,0.0
3,unknown,5514022,47.3
4,xrdcp,2652398,15.0
5,MakeReducedNtuple_NANO.x,2086857,0.0
6,python3,1364104,0.2
7,python3.12,1131981,0.0
8,xrootd,860109,0.6
9,python3.13,703067,0.0


**Evidence 2:** for each application, how many reporting hosts (with at least 1,000 flows) record almost always
server = destination (< 5 % inverted), almost always server = source (> 95 %), or a mix of the two.

In [10]:
hosts = q("""
WITH h AS (
    SELECT application, hostname, COUNT(*) AS n,
           AVG(CASE WHEN orient_class = 'S2C' THEN 1 ELSE 0 END) AS s
    FROM fo WHERE orient_class IN ('C2S', 'S2C')
    GROUP BY ALL HAVING COUNT(*) >= 1000)
SELECT application, COUNT(*) AS n_hosts,
       COUNT(*) FILTER (WHERE s < 0.05)                AS hosts_mostly_server_dst,
       COUNT(*) FILTER (WHERE s > 0.95)                AS hosts_mostly_server_src,
       COUNT(*) FILTER (WHERE s BETWEEN 0.05 AND 0.95) AS hosts_mixed
FROM h GROUP BY 1 ORDER BY n_hosts DESC LIMIT 15
""")
hosts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,application,n_hosts,hosts_mostly_server_dst,hosts_mostly_server_src,hosts_mixed
0,http-put,323,0,323,0
1,xrdcp,238,122,0,116
2,cmsRun,234,234,0,0
3,python3,227,226,0,1
4,JobWrapper,199,199,0,0
5,python3.12,140,140,0,0
6,python3.10,138,138,0,0
7,python3.11,134,134,0,0
8,unknown,109,1,0,108
9,alien.py/1.7.0,86,86,0,0


**Examples:** a reporting host with a mixed convention (if any): two flows per convention, same host and application.

In [11]:
mixed = q("""
SELECT application, hostname, COUNT(*) AS n,
       ROUND(100 * AVG(CASE WHEN orient_class = 'S2C' THEN 1 ELSE 0 END), 1) AS pct_server_is_source
FROM fo WHERE orient_class IN ('C2S', 'S2C')
GROUP BY ALL
HAVING COUNT(*) >= 1000
   AND AVG(CASE WHEN orient_class = 'S2C' THEN 1 ELSE 0 END) BETWEEN 0.05 AND 0.95
ORDER BY n DESC LIMIT 5
""")
display(mixed)

if len(mixed) > 0:
    app, host = mixed["application"].iloc[0], mixed["hostname"].iloc[0]
    display(pd.concat([
        examples("application = ? AND hostname = ? AND orient_class = 'C2S'", n=2, params=[app, host]),
        examples("application = ? AND hostname = ? AND orient_class = 'S2C'", n=2, params=[app, host]),
    ], ignore_index=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,application,hostname,n,pct_server_is_source
0,unknown,ceph-svc35.gridpp.rl.ac.uk,156195,49.8
1,unknown,ceph-svc37.gridpp.rl.ac.uk,155341,50.1
2,unknown,ceph-svc39.gridpp.rl.ac.uk,154959,49.9
3,unknown,ceph-svc40.gridpp.rl.ac.uk,154633,49.7
4,unknown,ceph-svc31.gridpp.rl.ac.uk,149464,49.8


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()
/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


,flow_id,t_start,src_ip,src_port,dst_ip,dst_port,application,hostname,src_site,dst_site
0,8efbe2990232ac623985babeed0daea0c0ac1013c9b965...,2026-10-10 22:00:36.503786+00:00,2001:630:54:12:82f6:db54::,40558,2001:630:54:20:82f6:b152::,1094,unknown,ceph-svc35.gridpp.rl.ac.uk,RAL-LCG2,RAL-LCG2
1,e85b139f22d84680840411c235cabf9a1ce331b93a54fc...,2026-10-14 09:47:32.738174+00:00,2001:630:58:1c20::82f6:2d9c,53042,2001:630:54:20:82f6:b152::,1094,unknown,ceph-svc35.gridpp.rl.ac.uk,UKI-SOUTHGRID-RALPP,RAL-LCG2
2,b3ac7e1045435adcd83920eab95a955a1422aa4f9e5da3...,2026-10-09 00:49:34.258754+00:00,2001:630:54:20:82f6:b152::,1094,2404:d540:1:589:2:3:1:503,56060,unknown,ceph-svc35.gridpp.rl.ac.uk,RAL-LCG2,TOKYO-LCG2
3,0146c2deb338c84faf94ea010155c630ee6efadb8589c5...,2026-10-09 00:35:55.904959+00:00,2001:630:54:20:82f6:b152::,1094,2405:3000:3:4:8::16,34396,unknown,ceph-svc35.gridpp.rl.ac.uk,RAL-LCG2,NaN


## Q3. Which endpoint is the server when the ports do not tell?
**Question for the developers:** in the flows below neither port, or both ports, is a known service port.
Which endpoint is the server? Are these, for example, third-party copies, data-channel connections or internal
transfers?

**Evidence:** flows by port configuration.

In [12]:
q("""
SELECT CASE
         WHEN src_svc AND dst_svc THEN 'both service ports'
         WHEN NOT src_svc AND NOT dst_svc AND src_port >= 32768 AND dst_port >= 32768 THEN 'both ephemeral (>= 32768)'
         WHEN NOT src_svc AND NOT dst_svc AND src_port <  32768 AND dst_port <  32768 THEN 'both below 32768'
         WHEN NOT src_svc AND NOT dst_svc THEN 'one below 32768, one ephemeral'
         ELSE 'exactly one service port' END AS port_case,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS pct
FROM fo GROUP BY 1 ORDER BY n DESC
""")

,port_case,n,pct
0,exactly one service port,66633079,99.091
1,both service ports,367930,0.547
2,both ephemeral (>= 32768),177298,0.264
3,"one below 32768, one ephemeral",62848,0.093
4,both below 32768,3364,0.005


**Examples:** three flows for each ambiguous case.

In [13]:
cases = {
    "both service ports":        "src_svc AND dst_svc",
    "both ephemeral (>= 32768)": "NOT src_svc AND NOT dst_svc AND src_port >= 32768 AND dst_port >= 32768",
    "both below 32768":          "NOT src_svc AND NOT dst_svc AND src_port < 32768 AND dst_port < 32768",
}
for name, where in cases.items():
    print(name)
    display(examples(where, n=3))

both service ports


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


,flow_id,t_start,src_ip,src_port,dst_ip,dst_port,application,hostname,src_site,dst_site
0,b531cd8a7e54e9ae12d780f4164572087bfbbce6ec46d6...,2026-10-10 22:50:47.236642+00:00,2607:f388:101c:1000::683,24128,2603:4000:c00:0:c:ffff:120c:1c1,1094,python3.12,t2dsk0027.cmsaf.mit.edu,GLOW,MIT_CMS
1,a2bb23a55f1c31b6c7afa4090df9222df492032eb1904f...,2026-10-08 09:51:12.918759+00:00,193.146.75.180,23220,128.104.227.192,1094,MakeReducedNtuple_NANO.x,g41n06.hep.wisc.edu,NaN,GLOW
2,0628bbcd5a360fb96ab6f7a76602e353670fff46d605eb...,2026-10-08 04:17:52.911687+00:00,2607:f388:101c:1000::309,21446,2001:1458:301:34::100:5d,1095,xrdcp,st-120-100gb-aduixt.cern.ch,GLOW,CERN-PROD


both ephemeral (>= 32768)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


,flow_id,t_start,src_ip,src_port,dst_ip,dst_port,application,hostname,src_site,dst_site
0,7d64585929f5cdd2a2dd2b3b2e445ef0c5c251a00ca45b...,2026-10-12 23:41:07.719370+00:00,2001:48a8:68f7:1:192:41:230:183,37424,2001:1458:301:101::100:d,44692,http-put,st-096-dd904b6a.cern.ch,AGLT2,CERN-PROD
1,8333c449f3d00b7d8f2a249baed6d159320dde9e9064f0...,2026-10-14 22:27:46.974521+00:00,2001:620:808:4015::39,35017,2001:630:40:ef::75,56302,http-put,cephc07.gla.scotgrid.ac.uk,NaN,UKI-SCOTGRID-GLASGOW
2,97c7d85c8f9e0349e6f9624e30291c12abc498a8793074...,2026-10-12 18:07:39.352790+00:00,2001:620:808:4015::38,35575,2001:660:3024:100:134:158:72:41,58832,http-put,grid-72041.ijclab.in2p3.fr,NaN,GRIF


both below 32768


/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


,flow_id,t_start,src_ip,src_port,dst_ip,dst_port,application,hostname,src_site,dst_site
0,d7c7a5a1ef9423e5cab482798363de2abf1d337be11acd...,2026-10-12 10:13:47.569254+00:00,2001:660:4705:d002:134:158:150:245,25000,2001:660:3031:110:20::44,18797,http-put,stor44.datagrid.cea.fr,IN2P3-IRES,GRIF
1,a66623cdbc74c7d4ce85eb91af6cca1e8848e62a8f6512...,2026-10-11 09:26:01.987681+00:00,2001:18e8:804:7::80d3:8f42,8003,2607:f388:101c:1000::81,26718,http-put,g39n01.hep.wisc.edu,Purdue,GLOW
2,f64bdc72699abbd8b286bab5bbb65225ec703563b011af...,2026-10-11 00:55:04.934503+00:00,2001:18e8:804:7::80d3:8f42,8002,2607:f388:101c:1000::52,14894,http-put,g33n03.hep.wisc.edu,Purdue,GLOW


Most frequent applications in each ambiguous case.

In [14]:
for name, where in cases.items():
    print(name)
    display(q(f"SELECT application, COUNT(*) AS n FROM fo WHERE {where} GROUP BY 1 ORDER BY n DESC LIMIT 5"))

both service ports


,application,n
0,cmsRun,99746
1,python3.12,47554
2,http-put,45786
3,MakeReducedNtuple_NANO.x,41201
4,JobWrapper,31484


both ephemeral (>= 32768)


,application,n
0,http-put,176746
1,http-tpc,552


both below 32768


,application,n
0,http-put,3363
1,unknown,1


## Q4. What are ports 20000–24999 and 30000–30999?
**Question for the developers:** are ports 20000–24999 and 30000–30999 server-side data ports (e.g. of storage
movers or a particular storage system)? Which software and sites use them?

**Evidence 1:** number of flows per range and per role of the port (destination or source).

In [15]:
RANGES = """(src_port BETWEEN 20000 AND 24999 OR dst_port BETWEEN 20000 AND 24999
          OR src_port BETWEEN 30000 AND 30999 OR dst_port BETWEEN 30000 AND 30999)"""

q(f"""
SELECT CASE WHEN dst_port BETWEEN 20000 AND 24999 OR src_port BETWEEN 20000 AND 24999
            THEN '20000-24999' ELSE '30000-30999' END AS port_range,
       CASE WHEN dst_port BETWEEN 20000 AND 24999 OR dst_port BETWEEN 30000 AND 30999
            THEN 'as destination' ELSE 'as source' END AS role,
       COUNT(*) AS n
FROM fo WHERE {RANGES}
GROUP BY ALL ORDER BY n DESC
""")

,port_range,role,n
0,20000-24999,as source,2533116
1,30000-30999,as source,406554
2,20000-24999,as destination,70497
3,30000-30999,as destination,8375


**Evidence 2:** sites and applications of the host that uses the port in these ranges.

In [16]:
q(f"""
SELECT CASE WHEN dst_port BETWEEN 20000 AND 24999 OR dst_port BETWEEN 30000 AND 30999
            THEN dst_site ELSE src_site END AS site_with_range_port,
       application, COUNT(*) AS n
FROM fo WHERE {RANGES}
GROUP BY ALL ORDER BY n DESC LIMIT 10
""")

,site_with_range_port,application,n
0,IN2P3-CC,http-put,371821
1,FZK-LCG2,http-put,256833
2,NaN,http-put,171599
3,TOKYO-LCG2,http-put,152811
4,MWT2,http-put,126474
5,pic,http-put,120489
6,ARNES,http-put,114797
7,TRIUMF-LCG2,http-put,113107
8,DESY-HH,http-put,110160
9,NET2,http-put,89438


**Examples:** two flows per range.

In [17]:
pd.concat([
    examples("src_port BETWEEN 20000 AND 24999 OR dst_port BETWEEN 20000 AND 24999", n=2),
    examples("src_port BETWEEN 30000 AND 30999 OR dst_port BETWEEN 30000 AND 30999", n=2),
], ignore_index=True)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


,flow_id,t_start,src_ip,src_port,dst_ip,dst_port,application,hostname,src_site,dst_site
0,0c004799a387333c19a3c654bf24b79f043db3a2501846...,2026-10-12 00:55:44.880914+00:00,2a00:139c:5:1d8:0:41:7d:a3,20425,2620:104:1f:1000:ba3f:d2ff:fe2d:6586,53984,http-put,cmsio5.ufhpc,FZK-LCG2,UFlorida-HPC
1,4cd0d9787bf9d2cc27b0fffa076e41b9158f9185c01909...,2026-10-09 12:10:46.368259+00:00,2001:48e8:6401:1::3033,22739,2001:630:80:2fd::aadb:8426,39368,http-put,door01.hec.lancs.ac.uk,NET2,UKI-NORTHGRID-LANCS-HEP
2,b5625ff6429a72ad9b2af9eecf1e087cfc9853c4567f79...,2026-10-12 03:32:25.867711+00:00,2001:660:5009:9:193:48:99:115,30894,2001:630:54:18:82f6:b39e::,39472,http-put,ceph-svc22.gridpp.rl.ac.uk,IN2P3-CC,RAL-LCG2
3,3147a99061963133d08fae29349c49ca31312b6d0c34da...,2026-10-12 09:56:40.182022+00:00,2001:660:5009:308:134:158:211:27,30311,2001:660:3031:110:20::41,50819,http-put,stor41.datagrid.cea.fr,IN2P3-CC,GRIF


## Q5. `end_rtt = 0` and USCMS-FNAL-WC1
**Question for the developers:** is `end_rtt = 0` a known measurement failure of the producer? Why is it
concentrated on flows involving USCMS-FNAL-WC1, and was it fixed during the week covered by the data?

**Evidence 1:** daily share of flows with `end_rtt = 0`, overall and for flows involving FNAL.

In [18]:
FNAL = "(src_site = 'USCMS-FNAL-WC1' OR dst_site = 'USCMS-FNAL-WC1')"

q(f"""
SELECT CAST(date_trunc('day', t_start) AS DATE) AS day,
       COUNT(*) AS n,
       ROUND(100 * AVG(CASE WHEN end_rtt = 0 THEN 1 ELSE 0 END), 3) AS pct_end_rtt_zero,
       COUNT(*) FILTER (WHERE {FNAL}) AS n_fnal,
       ROUND(100 * AVG(CASE WHEN end_rtt = 0 THEN 1 ELSE 0 END) FILTER (WHERE {FNAL}), 3) AS pct_end_rtt_zero_fnal
FROM fo GROUP BY 1 ORDER BY 1
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,day,n,pct_end_rtt_zero,n_fnal,pct_end_rtt_zero_fnal
0,2026-10-07,26,0.000,0,NaN
1,2026-10-08,10181245,0.245,90171,19.915
2,2026-10-09,10555038,0.773,322893,21.052
3,2026-10-10,9872362,1.403,420391,32.303
4,2026-10-11,9757811,0.974,252315,32.429
5,2026-10-12,9196187,0.177,76474,21.253
6,2026-10-13,8230756,0.011,199352,0.441
7,2026-10-14,9451094,0.002,296650,0.000


**Evidence 2:** share of all `end_rtt = 0` flows that involve FNAL, and the reporting hosts that record them most.

In [19]:
display(q(f"""
SELECT COUNT(*) AS n_end_rtt_zero,
       COUNT(*) FILTER (WHERE {FNAL}) AS n_fnal,
       ROUND(100.0 * COUNT(*) FILTER (WHERE {FNAL}) / COUNT(*), 1) AS pct_fnal
FROM fo WHERE end_rtt = 0
"""))

q("""
SELECT hostname, application, COUNT(*) AS n_end_rtt_zero
FROM fo WHERE end_rtt = 0
GROUP BY ALL ORDER BY n_end_rtt_zero DESC LIMIT 10
""")

,n_end_rtt_zero,n_fnal,pct_fnal
0,357462,320687,89.7


,hostname,application,n_end_rtt_zero
0,red-xfer1.unl.edu,xrdcp,50471
1,red-xfer2.unl.edu,xrdcp,50318
2,t2dsk0025.cmsaf.mit.edu,xrdcp,7973
3,t2dsk0015.cmsaf.mit.edu,xrdcp,7661
4,g40n06.hep.wisc.edu,xrdcp,6592
5,g42n06.hep.wisc.edu,xrdcp,6484
6,g32n06.hep.wisc.edu,xrdcp,6467
7,g33n06.hep.wisc.edu,xrdcp,6425
8,xrootd-05-prod-d7b7c655b-cpxbx,xrdcp,6401
9,xrootd-04-prod-864645db54-g7xth,xrdcp,6361


**Examples:** three flows involving FNAL with `end_rtt = 0`.

In [20]:
examples(f"end_rtt = 0 AND {FNAL}", n=3, extra=", start_rtt, end_rtt, t_end")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()
/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_57745/2129078219.py:52: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  return con.execute(sql, params or []).df()


,flow_id,t_start,src_ip,src_port,dst_ip,dst_port,application,hostname,src_site,dst_site,start_rtt,end_rtt,t_end
0,fe263bf98d357e8b436980e58e6ba6398da04fc1a32b71...,2026-10-11 20:38:28.522983+00:00,2603:4000:c00:0:c:ffff:120c:1c1,1094,2620:6a:0:8420:f0:0:207:98,60448,xrdcp,t2dsk0027.cmsaf.mit.edu,MIT_CMS,USCMS-FNAL-WC1,20.665,0.0,2026-10-11 20:38:47.983398+00:00
1,3f044a4fd1c34235303a0b023a6c6456deff99923e3e0f...,2026-10-11 17:18:49.235450+00:00,2603:4000:c00:0:c:ffff:120c:1ba,1094,2620:6a:0:8420:f0:0:207:163,40456,xrdcp,t2dsk0020.cmsaf.mit.edu,MIT_CMS,USCMS-FNAL-WC1,20.443,0.0,2026-10-11 17:20:52.442097+00:00
2,a20e81ddf29e91d78829e20f273a30b14d8da4b3c56401...,2026-10-10 08:20:14.977212+00:00,2607:f388:101c:1000::638,1094,2620:6a:0:8420:f0:0:206:252,38348,xrdcp,g40n06.hep.wisc.edu,GLOW,USCMS-FNAL-WC1,5.886,0.0,2026-10-10 08:20:15.867817+00:00
